# Определение стоимости автомобилей

Сервис по продаже автомобилей с пробегом «Не бит, не крашен» разрабатывает приложение для привлечения новых клиентов. В нём можно быстро узнать рыночную стоимость своего автомобиля. В вашем распоряжении исторические данные: технические характеристики, комплектации и цены автомобилей. Вам нужно построить модель для определения стоимости. 

Заказчику важны:

- качество предсказания;
- скорость предсказания;
- время обучения.

## Настройка окружения

### Импорты

In [ ]:
# %pip install -Uq numpy matplotlib contourpy scikit-learn scipy pandas pyarrow
# %pip install -Uq plotly
# %pip install -Uq shap
%pip install -Uq phik
# %pip install -Uq lightgbm
# %pip install -Uq catboost

In [ ]:
# imports
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

import warnings

warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor  # type: ignore
from lightgbm import LGBMRegressor  # type: ignore
from phik import phik_matrix
from sklearn.metrics import (
    mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
)

from sklearn.tree import DecisionTreeRegressor

### Настройки отображения

In [ ]:
# output settings
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_colwidth', None)

### Объявление функций

In [ ]:
def check_size(current_df: pd.DataFrame, original_df: pd.DataFrame):
    print("Количество записей в текущем датасете: {}".format(len(current_df)))
    print("Количество записей в оригинальном датасете: {}".format(len(original_df)))
    print("Процент от начального объема данных: {:.2%}".format(len(current_df) / len(original_df)))

In [ ]:
import os

HOST = "https://code.s3.yandex.net"
HTTP_PREFIX = "http"


# load csv
def load_csv(dataset_path: str, **kwargs):
    # check server request --> relative path --> absolute path --> yandex server request
    path = (
        dataset_path
        if dataset_path.startswith(HTTP_PREFIX)
        else "." + dataset_path if os.path.exists("." + dataset_path)
        else dataset_path if os.path.exists(dataset_path)
        else HOST + dataset_path
    )
    print("Dataset path:", path)
    try:
        return pd.read_csv(filepath_or_buffer=path, **kwargs)
    except Exception as ex:
        print("Could not load csv. Exception:", str(ex))

In [ ]:
def df_init_analysis(df_name, df):
    print("=" * 50)
    print(f"{df_name}\n")
    print("Общая информация\n")
    print(df.info())

    print("\nБазовая статистика по данным")
    display(df.describe(include='all'))

    print("\nИнформация по колонкам\n")
    df_init_analysis_column_names = [
        "column_name",
        "type",
        "na_count",
        "empty_count",
        "unique_count",
    ]

    df_init_analysis_data = []
    for column_name in df.columns.tolist():
        df_init_column_data = []
        df_init_column_data.append(column_name)
        df_init_column_data.append(df[column_name].dtype)
        df_init_column_data.append(df[column_name].isna().sum())
        df_init_column_data.append(sum(df[column_name] == ""))
        df_init_column_data.append(df[column_name].nunique())
        df_init_analysis_data.append(df_init_column_data)

    df_init = pd.DataFrame(columns=df_init_analysis_column_names, data=df_init_analysis_data)
    display(df_init)

    print(f"\nКоличество дубликатов: {df.duplicated().sum()}\n")

In [ ]:
def camel_to_snake(column_name):
    """Преобразует название столбца из CamelCase в snake_case"""
    return ''.join(['_' + char.lower() if char.isupper() else char for char in column_name]).lstrip('_')


def rename_columns_to_snake_case(df):
    """Переименовывает столбцы DataFrame из CamelCase в snake_case"""
    df.columns = [camel_to_snake(col) for col in df.columns]
    return df

## Загрузка данных

In [ ]:
df_autos_original = load_csv("/datasets/autos.csv")
df_autos = df_autos_original.copy()
df_autos.head()

## Предобработка данных

### Первичный анализ данных

In [ ]:
df_dict = {
    "df_autos": df_autos,
}

In [ ]:
for df_name, df in df_dict.items():
    df_init_analysis(df_name, df)

Данные содержат 354369 записи.

В данных присутствуют пропуски, выбросы и некорректные значения. Пропуски содержатся только в категориальных признаках. Некорректно определен тип данных для признаков с датами. Также в данных есть неинформативные признаки и признаки, которые имеет смысл преобразовать в новые признаки.

Примеры:
- пропуски: VehicleType, Gearbox, Model, FuelType, Repaired
- выбросы: Price. Максимальное значение 20,000 похоже на выброс, т.к. для 75% объявлений это значение меньше 6,400
- некорректные значения: RegistrationYear (максимальное значение 9999), Power (максимальное значение 20,000), RegistrationMonth (минимальное значение 0)
- тип данных: DateCrawled, DateCreated, LastSeen
- неинформативные признаки: DateCrawled, RegistrationMonth, DateCreated, NumberOfPictures, PostalCode, LastSeen

Также присутствуют дубликаты в количестве 4 штук.


### Переименование столбцов

In [ ]:
# Применяем функцию переименования
df_autos = rename_columns_to_snake_case(df_autos)

print("После переименования:\n")
print(df_autos.info())

### Удаление неинформативных признаков

In [ ]:
cols_to_drop = [
    "date_crawled",
    "registration_month",
    "date_created",
    "number_of_pictures",
    "postal_code",
    "last_seen",
]


df_autos = df_autos.drop(columns=cols_to_drop)
df_autos.info()

### Обработка пропусков

Найдем и выведем общее количество строк с пропущенными значениями

In [ ]:
def print_df_na(df_name, df):
    # Создаем маску для строк с NaN
    mask = df.isna().any(axis=1)
    if mask.sum() > 0:
        # Выводим строки с пропущенными значениями
        # rows_with_na = df.loc[mask]
        print("-" * 20)
        print(f"{df_name}:\n")
        print("Общее количество строк с пропущенными значениями:", mask.sum())
        # display(rows_with_na)

In [ ]:
for df_name, df in df_dict.items():
    print_df_na(df_name, df)

Данных с пропусками много. Удалять их нельзя. Обработаем пропуски. Поскольку пропуски содержатся только в категориальных признаках, заполним их значением `other`.

In [ ]:
df_autos = df_autos.fillna(value='other')

In [ ]:
print("После заполнения пропусков:\n")
print(df_autos.info())

In [ ]:
df_autos.head()

### Преобразование типов данных

Преобразуем категориальные признаки в тип category.

In [ ]:
cat_cols = [
    'vehicle_type',
    'gearbox',
    'model',
    'fuel_type',
    'brand',
    'repaired',
]

for col in cat_cols:
    df_autos[col] = df_autos[col].astype('category')

df_autos.info()

### Наличие дубликатов

Проверим наличие неявных дубликатов.

In [ ]:
def find_implicit_duplicates(df, cat_cols=None):
    """
    Функция находит неявные дубликаты в категориальных столбцах DataFrame.
    Выводит пары возможных неявных дубликатов.
    """
    # Получаем список категориальных столбцов (строкового типа), если cat_cols не задано
    if cat_cols is None:
        cat_cols = df.select_dtypes(include=['object', 'string']).columns

    for col in cat_cols:
        unique_values = df[col].unique()
        normalized_values = {val.strip().lower() for val in unique_values}

        if len(unique_values) != len(normalized_values):
            print("-" * 20)
            print(f"\nСтолбец '{col}' содержит неявные дубликаты:\n")

            # Строим словарь соответствия оригинальных значений и нормализованных
            mapping = {}
            for original_value in unique_values:
                norm_val = original_value.strip().lower()
                mapping.setdefault(norm_val, []).append(original_value)

            # Показываем группы потенциально одинаковых значений
            for key, group in mapping.items():
                if len(group) > 1:
                    print(f"Подозрительная группа значений ({key}): {group}\n")


find_implicit_duplicates(df_autos, cat_cols)

Отсортируем и выведем все уникальные значения категориальных признаков

In [ ]:
for col in df_autos.select_dtypes(include=['category']).columns:
    unique_values = sorted(df_autos[col].unique())
    print(f'{col}: {unique_values}')

ОБнаружились неявные дубли в названиях моделей: 
- 'range_rover', 'rangerover'
- ('1_reihe', '1er', 'serie_1'), ('2_reihe', 'serie_2'), ('3_reihe', '3er', 'serie_3') и подобные -- похожи на немецкие названия серий BMW. Убедимся в этом.
- 'gasoline' и 'petrol' -- это одно и то же. Разные термины употребляются в разных странах для обозначения бензина.

In [ ]:
brand = 'bmw'
bmw_models = [
    '1_reihe', '1er', 'serie_1', '2_reihe', 'serie_2', '3_reihe', '3er', 'serie_3'
]

mask = (df_autos['brand'] != brand) & (df_autos['model'].isin(bmw_models))
df_autos[mask].head(10)

Не только bmw. У mazda и peugeot тоже есть серии. А 'serie_1', 'serie_2' и 'serie_3' встречаются у land_rover.

Переименуем 'rangerover' в 'range_rover', а 'gasoline' в 'petrol'.

In [ ]:
df_autos['model'] = df_autos['model'].replace('rangerover', 'range_rover')
df_autos['fuel_type'] = df_autos['fuel_type'].replace('gasoline', 'petrol')

In [ ]:
for col in df_autos.select_dtypes(include=['category']).columns:
    unique_values = sorted(df_autos[col].unique())
    print(f'{col}: {unique_values}')

Неявные дубликаты ушли.

Проверим наличие явных дубликатов

In [ ]:
def print_df_duplicates(df_name, df):
    # Создаем маску для строк с дубликатами
    mask = df.duplicated()
    if mask.sum() > 0:
        # Выводим дубликаты
        print("-" * 20)
        print(f"\n{df_name}: total: {mask.sum()}\n")
        display(df[mask].head())
    return mask.sum()

In [ ]:
df_dict = {
    "df_autos": df_autos,
}

duplicates_count = 0
for df_name, df in df_dict.items():
    duplicates_count += print_df_duplicates(df_name, df)
if duplicates_count == 0:
    print("Явных дубликтов не обнаружено")

После всех преобразований количество явных дубликатов увеличилось -- всего 46042 записи. Для последующего обучения моделей удалим их.

In [ ]:
df_autos = df_autos.drop_duplicates()

### Создание новых признаков

In [ ]:
# date_cols = [
#     'date_created',
# ]

# for col in date_cols:
#     df_autos[col] = df_autos[col].apply(pd.to_datetime, errors='raise')

# df_autos.info()

In [ ]:
# df_autos['created_year'] = df_autos['date_created'].dt.year
# df_autos['created_month'] = df_autos['date_created'].dt.month
# df_autos.info()

In [ ]:
df_autos.info()

### Полнота данных

Проверим полноту данных после проведения предобработки

In [ ]:
check_size(df_autos, df_autos_original)

### Промежуточные выводы

В ходе предварительной обработки данных было сделано следующее:
- проведена проверка на наличие пропусков в данных, пропуски обнаружены, заполнены значение `other`
- проведена проверка на наличие явных и неявных дубликатов. Неявные дубликаты приведены к единому виду, явные дубликаты удалены
- удаление неинформативных признаков
- проведена проверка на полноту данных после предобработки

## Исследовательский анализ данных

### Количественные признаки

In [ ]:
# количество корзин в зависимости от количества уникальных значений
def get_bins_count(value):
    print(f"n_uniq_values: {value}")
    n_bins = value // 100 if value > 1000 else value // 10 if value > 100 else value
    print(f"n_bins: {n_bins}")
    return n_bins

In [ ]:
# histogram function
def get_histogram(df, df_name, df_column_name):
    fig = px.histogram(
        df,
        x=df_column_name,
        nbins=get_bins_count(df[df_column_name].nunique()),
        marginal='box',
        barmode="group",
        opacity=0.5,
        title=f'Распределение значений {df_name}.{df_column_name}',
    )

    fig.update_layout(
        xaxis_title_text=f"{df_name}.{df_column_name}",
        yaxis_title_text="Кол-во",
    )

    fig.show()

In [ ]:
num_cols = df_autos.select_dtypes(include=['number']).columns

df_dict = {
    "df_autos": df_autos,
}

for df_name, df in df_dict.items():
    for column_name in num_cols:
        get_histogram(df, df_name, column_name)

Наблюдаются явные ошибки в данных

Год регистрации автомобиля не может быть раньше 1900 и больше 2016. Регистрация автомобилей до 1900 была единичными случаями. А максимальный год размещения объявлений (2016) говорит о том, что автомобили не могли быть зарегестрированы после него.

Основными единицами измерения мощности автомобиля являются лошадиные силы и киловатты. Вероятно, часть данных указана в лошадиных силах, а часть в киловаттах. Но даже если предположить, что значение 20,000 указано в Вт или кВт, то это будет примерно 27 или 27,000 лошадиных сил, что также выглядит сомнительно. Минимально приемлемая мощность легковых автомобилей массового производства обычно начинается от 20 л.с., поскольку меньшие значения недостаточны для эффективного функционирования транспортного средства на дорогах общего пользования. А максимально известная мощность автомобиля в лошадиных силах составляет 2300 лошадиных сил, которую обеспечивает модель Koenigsegg Gemera. Судя по гистограмме распределения мощности, подавляющая масса автомобилей находится в пределах 500 л.с. Возьмем эту цифру за основу. Все остальное будем считать выбросами. Если посмотреть на этот признак с точки зрения здравой логики, то машины мощностью выше 500 л.с. встречаются редко.

Рассмотрим эти аномалии детальней.

Рассмотрим автомобили с указанным годом регистрации до 1900 и после 2016

In [ ]:
df_autos_registration_year = df_autos.query('registration_year < 1900 or registration_year > 2016')
df_autos_registration_year.head(10)

In [ ]:
check_size(df_autos_registration_year, df_autos_original)

Посмотрим на гистограмму распределения по годам выпуска с другим значением корзин.

In [ ]:
# histogram function
def get_histogram(df, df_name, df_column_name):
    fig = px.histogram(
        df,
        x=df_column_name,
        nbins=20,
        marginal='box',
        barmode="group",
        opacity=0.5,
        title=f'Распределение значений {df_name}.{df_column_name}',
    )

    fig.update_layout(
        xaxis_title_text=f"{df_name}.{df_column_name}",
        yaxis_title_text="Кол-во",
    )

    fig.show()

df_autos_registration_year = df_autos.query('registration_year >= 1900 and registration_year <= 2016')
get_histogram(df_autos_registration_year, 'df_autos', 'registration_year')

Подавляющее большинство автомобилей зарегестрированы после 1990 года. В период 1970-1990 были зарегестрированы примерно 8К автомобилей. Немало, сохраним их. Все, что раньше 1970 будем считать выбросами.

Рассмотрим объявления с указанной мощностью меньше 30 или больше 500

In [ ]:
df_autos_power = df_autos.query('power < 30 or power > 500')
df_autos_power.head(10)

In [ ]:
check_size(df_autos_power, df_autos_original)

In [ ]:
df_autos_power = df_autos.query('power >= 30 and power <= 500')
get_histogram(df_autos_power, 'df_autos', 'power')

Гистограмма распределения по признаку power говорит о том, что даже выше 400 уже можно считать выбросами или аномалиями. Но оставим верхнюю границу на уровне 500.

Объем некорректных данных довольно высок и составляет больше 10% от общего объема данных.

Но вместе с тем, год регистрации (выпуска) автомобиля и его мощность оказывают сильное влияние на целевой признак -- стоимость автомобиля. И поскольку качество моделей и их предсказания имеет высокую важность, такие некорректные данные имеет смысл удалить.

In [ ]:
df_autos_to_drop = df_autos.query('(power < 30 or power > 500) or (registration_year < 1970 or registration_year > 2016)')
df_autos = df_autos.drop(df_autos_to_drop.index)

Посмотрим также на цену в границах от 0 до 1000 евро. Выглядит маловероятным, что автомобили раздают (почти) бесплатно.

In [ ]:
df_autos_low_price = df_autos.query('price <= 1000')
get_histogram(df_autos_low_price, 'df_autos', 'price')

И отдельно ниже 100 евро

In [ ]:
df_autos_low_price = df_autos.query('price <= 100')
get_histogram(df_autos_low_price, 'df_autos', 'price')

Похоже на ошибки в данных. Удалим все, что ниже 100 евро.

In [ ]:
df_autos = df_autos.query('price >= 100')

In [ ]:
check_size(df_autos, df_autos_original)

### Категориальные признаки

In [ ]:
# функция по отрисовке столбчатой и круговой диаграммы
def show_bar_pie(df, df_cat_column_name):
    count_column = df.columns.to_list()[0]
    # Сгруппируем данные по признаку
    df = df.groupby(by=[df_cat_column_name], observed=True)[count_column].count().reset_index().set_axis(
        [df_cat_column_name, "count"], axis=1)
    display(df)
    print()

    # Столбчатая диаграмма
    fig = px.bar(
        df,
        x=df[df_cat_column_name],
        y=df["count"],
    )
    fig.show()
    print()

    # Круговая диаграмма
    fig = px.pie(
        df,
        names=df[df_cat_column_name],
        values=df["count"],
    )
    
    fig.show()
    print()

In [ ]:
# функция по отрисовке столбчатой (для отдельных признаков)
def show_bar(df, df_cat_column_name):
    count_column = df.columns.to_list()[0]
    # Сгруппируем данные по признаку
    df = df.groupby(by=[df_cat_column_name], observed=True)[count_column].count().reset_index().set_axis(
        [df_cat_column_name, "count"], axis=1)
    # display(df)
    print()

    # Столбчатая диаграмма
    fig = px.bar(
        df,
        x=df[df_cat_column_name],
        y=df["count"],
    )
    fig.show()
    print()

In [ ]:
col_list = [
    'model',
    'brand',
]

for column_name in col_list:
    print("-" * 50)
    print(f"Диаграммы {column_name}")
    show_bar(df_autos, column_name)

In [ ]:
cat_cols = [
    'vehicle_type',
    'gearbox',
    # 'model',
    'fuel_type',
    # 'brand',
    'repaired',
]

for cat_column_name in cat_cols:
    print("-" * 50)
    print(f"Диаграммы {cat_column_name}")
    show_bar_pie(df_autos, cat_column_name)

### Промежуточные выводы

Больше всего объявлений о продаже автомобилей:
- бренда VW
- с ручной коробкой передач
- бензиновым двигателем
- без ремонта
- седанов

В существенной части объявлений не указана конкретная модель. На втором месте Golf.

## Корреляционный анализ

In [ ]:
# выделим колонки с количественными и категориальными данными
num_columns = list(df_autos.select_dtypes(include="number").columns)
cat_columns = list(df_autos.select_dtypes(exclude="number").columns)

# построим матрицу корреляции
correlation_matrix = df_autos.phik_matrix(interval_cols=num_columns, verbose=False)

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f');

Наблюдается мультиколлинеарность признаков (с коэффициентом зависимости больше 0.9):
- brand и model
- vehicle_type и model

Все вполне объяснимы. Каждая модель принадлежит только одному бренду, и нередко модели называются по-разному, в зависимости от типа кузова.

Судя по матрице корреляции, целевой признак (`price`) слабо зависит от следующих признаков (коэффициент зависимости меньше либо равен 0.3):
- vehicle_type
- gearbox
- fuel_type

Умеренно либо сильно зависит от:
- registration_year
- power
- model
- kilometer
- brand
- repaired

Избавимся от мультиколлинеарности признаков, удалив признак brand.

In [ ]:
cols_to_drop = [
    "brand",
]

df_autos = df_autos.drop(columns=cols_to_drop)
df_autos.info()

### Промежуточные выводы

Матрица корреляции показала заметную или сильную зависимость целевого признака (price) от года выпуска автомобиля и конкретной модели. В меньшей степени от остальных признаков. Это выглядит вполне логично и дополнительно подтверждает выводы, сделанные на предыдущих шагах анализа.

После удаления всех аномалий и дублей в датасете осталось 71.72% от изначального объема данных.

## Подготовка данных

In [ ]:
df_autos.info()

In [ ]:
RANDON_STATE = 1
TEST_SIZE = 0.25

# Отделяем признаки и цель
X = df_autos.drop("price", axis=1)
y = df_autos["price"]

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDON_STATE
)

# Определение категориальных и числовых признаков
numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(exclude=["number"]).columns

# Трансформация данных с помощью пайплайна
preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            # OneHotEncoder(handle_unknown="ignore", sparse=False),
            categorical_features,
        ),
    ]
)

## Обучение моделей

### DecisionTreeRegressor

In [ ]:
# Пайплайн с деревом решений
tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("regression", DecisionTreeRegressor(random_state=RANDON_STATE)),
    ]
)

# Диапазоны гиперпараметров для случайного поиска
param_dist = {
    "regression__max_depth": [None, 1, 5, 10],
    "regression__min_samples_leaf": [1, 2, 4, 8],
    # "regression__min_samples_split": [2, 5, 10, 15],
    # "regression__criterion": ["squared_error", "absolute_error"],
}

# Случайный поиск гиперпараметров
random_search = RandomizedSearchCV(
    estimator=tree_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=RANDON_STATE,
)

In [ ]:
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print(f"Лучшие гиперпараметры: {best_params}")

# Средняя метрика для лучшей модели
best_score = random_search.best_score_
print(f'Средняя метрика RMSE на кросс-валидации для лучшей модели: {-best_score:.2f}')

### LightGBM

In [ ]:
# Пайплайн с LightGBM
lgbm_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regression', LGBMRegressor(random_state=RANDON_STATE, verbose=-1))
])

# Диапазоны гиперпараметров для случайного поиска
param_dist = {
    'regression__learning_rate': [0.01, 0.1],
    'regression__n_estimators': [10, 20, 30],
    'regression__max_depth': [-1, 5, 10],
    'regression__num_leaves': [63, 127],
    'regression__min_child_samples': [5, 10],
}

# Случайный поиск гиперпараметров
random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDON_STATE
)

In [ ]:
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print(f'Лучшие гиперпараметры: {best_params}')

# Средняя метрика для лучшей модели
best_score = random_search.best_score_
print(f'Средняя метрика RMSE на кросс-валидации для лучшей модели: {-best_score:.2f}')

### CatBoost

In [ ]:
# CatBoostRegressor
cb_regressor = CatBoostRegressor(cat_features=categorical_features.values, random_seed=RANDON_STATE, verbose=0)

# Диапазоны гиперпараметров для случайного поиска
param_dist = {
    'iterations': [10, 20, 30],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [128, 254],
}

# Случайный поиск гиперпараметров
random_search = RandomizedSearchCV(
    estimator=cb_regressor,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=RANDON_STATE
)

In [ ]:
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print(f'Лучшие гиперпараметры: {best_params}')

# Средняя метрика для лучшей модели
best_score = random_search.best_score_
print(f'Средняя метрика RMSE на кросс-валидации для лучшей модели: {-best_score:.2f}')

Таким образом, определили 3 лучшие модели с параметрами для анализа:
- DecisionTreeRegressor: {'regression__min_samples_leaf': 8, 'regression__max_depth': None}
- LGBMRegressor: {'regression__num_leaves': 127, 'regression__n_estimators': 20, 'regression__min_child_samples': 5, 'regression__max_depth': -1, 'regression__learning_rate': 0.1}
- CatBoostRegressor: {'learning_rate': 0.2, 'l2_leaf_reg': 3, 'iterations': 30, 'depth': 8, 'border_count': 128}

## Анализ моделей

### DecisionTreeRegressor

In [ ]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "regression",
            DecisionTreeRegressor(min_samples_leaf=8, random_state=RANDON_STATE),
        ),
    ]
)

print("Обучение модели:")
%time tree_pipeline.fit(X_train, y_train)

print("Предсказание модели:")
%time tree_y_pred = tree_pipeline.predict(X_test)

# rmse = mean_squared_error(y_test, y_pred) ** 0.5
# print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')

### LGBMRegressor

In [ ]:
lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "regression",
            LGBMRegressor(
                num_leaves=127,
                n_estimators=20,
                min_child_samples=5,
                max_depth=-1,
                learning_rate=0.1,
                random_state=RANDON_STATE,
                verbose=-1,
            ),
        ),
    ]
)

print("Обучение модели:")
%time lgbm_pipeline.fit(X_train, y_train)

print("Предсказание модели:")
%time lgbm_y_pred = lgbm_pipeline.predict(X_test)

# rmse = mean_squared_error(y_test, y_pred) ** 0.5
# print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')

### CatBoostRegressor

In [ ]:
cb_regressor = CatBoostRegressor(
    cat_features=categorical_features.values, 
    learning_rate=0.2,
    l2_leaf_reg=3,
    iterations=30,
    depth=8,
    border_count=128,
    random_seed=RANDON_STATE, 
    verbose=0
)

print("Обучение модели:")
%time cb_regressor.fit(X_train, y_train)

print("Предсказание модели:")
%time cb_y_pred = cb_regressor.predict(X_test)

# rmse = mean_squared_error(y_test, y_pred) ** 0.5
# print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')

| Model | Fit | Predict | RMSE |
|---|---|---|---|
| DecisionTreeRegressor | 4.33 s | 151 ms | 1839.18 |
| LGBMRegressor | 995 ms | 253 ms | 1947.76 |
| CatBoostRegressor | 1.12 ms | 12.7 ms | 1840.74 |

Заказчик выдвинул требования к моделям:
- качество предсказания
- время обучения модели
- время предсказания модели

Качество предсказания:
1. DecisionTreeRegressor
2. CatBoostRegressor

Время обучения модели:
1. LGBMRegressor
2. CatBoostRegressor

Время предсказания модели:
1. CatBoostRegressor
2. DecisionTreeRegressor

По совокупности факторов CatBoostRegressor признана самой подходящей под требования заказчика. Качество предсказаний на кросс-валидации и времени обучения модели не сильно уступает лидерам, а по скорости предсказаний занимает первое место с большим отрывом.

### Тестирование лучшей модели

In [ ]:
rmse = mean_squared_error(y_test, cb_y_pred) ** 0.5
print(f'Root Mean Squared Error (RMSE): {rmse:.2f}')

Согласно требованиям, значение метрики RMSE должно быть меньше 2500. CatBoostRegressor показала значение 1827.93 на тестовой выборке, что соответствует требованиям.

Показанное на тестовой выборке значение RMSE лучше, чем на кросс-валидации, и лучше, чем значение на кросс-валидации для модели DecisionTreeRegressor.

## Общий вывод

В текущей работе было проведено исследование данных из объявлений о продажах автомобилей в Европе с целью построить модель, которая умеет определять стоимость подержанных автомобилей на основании данных об их технических характеристиках и комплектации.

Были обучены несколько различных моделей регрессии для сравнения характеристик моделей: время обучения, время предсказания, точность результата.

На основании проведенного исследования была выявлена лучшая модель по заданным требованиям. Ей стала CatBoostRegressor.

## Чек-лист проверки

Поставьте 'x' в выполненных пунктах. Далее нажмите Shift+Enter.

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Выполнена загрузка и подготовка данных
- [x]  Выполнено обучение моделей
- [x]  Есть анализ скорости работы и качества моделей